In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import colorir as cl
from analyses.rust import contact
from analyses import io
from pathlib import Path

In [ ]:
contactdf = pl.read_parquet("../runs/cil/**/act_contacts/*.parquet", include_file_paths="path") \
    .with_columns(list=pl.col("path").str.split("/")) \
    .with_columns(
        gamma=20 - pl.col("list").list.get(-4).str.replace("energy-", "").cast(pl.UInt32),
        time=pl.col("list").list.get(-1).str.replace(".parquet", "").cast(pl.UInt32),
        spin=pl.col("spin").cast(pl.UInt32),
        cell_cell=~(pl.col("neigh") == "m")
    ) \
    .filter(
        pl.col("time") > 5e5,
        pl.col("neigh") != "s"
    ) \
    .select(pl.exclude("path", "list")) \
    .sort("gamma", "time", "spin")
contactdf

In [ ]:
aggdf = (
    contactdf
        .group_by("gamma", "cell_cell")
        .agg(
            pl.len(),
            pl.mean("act"),
            pl.mean("kact"),
            std_act=pl.std("act"),
            std_kact=pl.std("kact")
        )
        .with_columns(
            ci_act=1.96 * pl.col("std_act") / pl.col("len").sqrt(),
            ci_kact=1.96 * pl.col("std_kact") / pl.col("len").sqrt()
        )
        # .unpivot(on=["act", "kact"], index=["gamma", "cell_cell"])
        .sort("gamma", "cell_cell")
)
aggdf

In [ ]:
aggdf["len"].min()

In [ ]:
colors = cl.StackPalette.load("safe")
fig = px.bar(
    aggdf,
    x="gamma",
    y="act", 
    #error_y="ci_act",
    color="cell_cell",
    barmode="overlay",
    color_discrete_sequence=colors
)
fig.update_layout(
    template="plotly_white",
    width=500,
    height=300,
    yaxis_title="mean act",
    bargap=0.,
)
fig.update_traces(marker_opacity=1)
io.save_plot(fig, "../plots/contact_act-gamma")

In [ ]:
fig = px.bar(
    aggdf,
    x="gamma",
    y="kact",
    # error_y="ci_kact",
    color="cell_cell",
    barmode="overlay",
    color_discrete_sequence=colors
)

# Confidence intervals:
# meddf = aggdf.filter(cell_cell=False)
# fig.add_traces(px.scatter(
#     x=meddf["gamma"],
#     y=meddf["kact"] - meddf["ci_kact"], 
#     symbol_sequence=["line-ew-open"]
# ).update_traces(marker_size=9).data)
# cdf = aggdf.filter(cell_cell=True)
# fig.add_traces(px.scatter(
#     x=cdf["gamma"],
#     y=cdf["kact"] + cdf["ci_kact"],
#     symbol_sequence=["line-ew-open"]
# ).update_traces(marker_size=9, marker_color="#bb0000").data)

fig.update_layout(
    template="plotly_white",
    width=500,
    height=300,
    yaxis_title="mean effective act",
    bargap=0.
)
fig.update_traces(marker_opacity=1)
io.save_plot(fig, "../plots/contact_kact-gamma")

In [ ]:
aggdf = contactdf \
    .filter(gamma=12) \
    .group_by("gamma", "cell_cell") \
    .mean() \
    .select(pl.exclude("time", "spin", "neigh")) \
    .unpivot(index=["gamma", "cell_cell"]) \
    .pivot(on="cell_cell", index="variable", values="value") \
    .with_columns(
        diff=pl.col("false") - pl.col("true"),
        mdiff=pl.col("false") / pl.col("true")
    ) \
    .unpivot(index="variable", variable_name="column")
aggdf

In [ ]:
fig = px.bar(aggdf, x="variable", y="value", color="column", barmode="group")
fig.update_layout(
    template="plotly_white",
    width=600,
    height=400,
    xaxis_title="kernel",
    yaxis_title="mean act"
)
io.save_plot(fig, "../plots/act_kernel")

In [ ]:
klats = []
clat_path = "../runs/cil/energy-8/lattices/0001000000.parquet"
alat_path = "../runs/cil/energy-8/act_lattices/0001000000.parquet"
clat, alat = pl.read_parquet(clat_path), pl.read_parquet(alat_path)
for r in range(0, 17):
    res = contact.kernel_act(clat, alat, r, True)
    klat = np.array(res[1]).reshape(500, 500)
    klats.append(klat)
klat_3d = np.array(klats)

In [ ]:
fig = px.imshow(
    np.where(np.isin(clat.to_numpy().T, ["s", "m"]), np.nan, klat_3d[0:]), 
    animation_frame=0,
    width=800,
    height=800,
    zmax=20,
    zmin=0,
    color_continuous_scale=cl.PolarGrad(["#298d80", "#fa9f6a"]).to_plotly_colorscale()
)
fig.update_layout(
    template="plotly_white",
    xaxis_range=[100, 400],
    yaxis_range=[100, 400], 
    yaxis_autorange=False,
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    xaxis_constrain="domain",
    xaxis_visible=False,
    yaxis_visible=False,
    margin=dict(l=0, r=0, t=0, b=0),
)
# io.save_plot(fig, "../plots/act_kernel_img_1")